# AP 155 Lab Exercise
---
## Exercise 5: Matrices

_Instructions_: 
- **Report figure:** No report figure. Points in the report figure will be allocated to the report discussion 
- **Report discussion:**
  - What were the basis linear equations used to set up the matrix in each question of Problem 2?
  - Once you set up the matrices, what function did you use to solve the matrix equation (Could have different answers)? Check the docstring/documentation/other references and summarize the mechanism behind the function 
- **Code**: Complete the code and ensure it is functional, error-free, readable, and efficient (where needed). Include concise Markdown documentation highlighting the critical steps of your algorithm for each problem.

### Student Information
- _Full Name (Last Name, First Name)_: Iniego, Angel Nicole
- _Student No._: 2024-09622
- _Section_: THR-TX-X

### Grading Information (c/o Lab Instructor)
- [Rubrics description link](https://drive.google.com/file/d/1BMSlPot2Mc7XLu0eo4S8I8gLBsIadbCL/view?usp=sharing) (Note: percentages may still be tweaked)

| Criteria | Score | Subtotal |
| --- | --- | --- |
| Report figure | XX | 20 |
| Report discussion | XX | 20|
| Code readability | XX | 20 |
| Code efficiency | XX | 20 | 
| Code appropriateness  | XX | 20 | 
| **TOTAL** | XXX | 100 |

_Date and Time Scored (MM/DD/YYYY HH:MM AM/PM):_:_

---
## Section 1: Report

<!-- ### Report figure

<img src="figures/cat.jpg" alt="Alt text" width=45%> <img src="figures/cat.jpg" alt="Alt text" width=45%>


EDIT ME. See instruction above for what to include. Insert caption here, a short description what the figure illustrates. The syntax in markdown for putting a figure is `![Description](local_file_name.png)` or `<img src="/path/to_image.jpg" alt="Alt text" width=45%>`. Make sure the figure has complete elements.
-->

### Report discussion
For n = 6, I used np.linalg.solve as suggested in class. According to the official documentation, the matrices are stored in contiguous memory which means the computer saves it as a single 1d list where the end of one row connects directly to the start if the next row. It then uses Gaussian elimination with row interchanges to factor the A into a permutation matrix, a unit lower triangular matrix L, and an upper triangular matrix U, as demonstrated in class. This results in a time O(N^3) since every entry, including the zeroes, is visited and modified during elimination.

However, for the larger matrix with size 10000 x 10000, I used the scipy.sparse, which is generally recommended by many users (see <href> https://github.com/numpy/numpy/issues/23829 </href>). The documentation says the matrices are compressed into Compressed Sparse Column (CSC) where only non-zero values and their indices are stored. It then does a Sparse LU which even monitors the zero entries that unintentinally turn non-zero during Gaussian elimination, then sparse factorization. This results in an O(N) complexity for which N are the non-zero elements. 

For Problem 2 Part 1, the systems are solved by applying the matrix which only obtains the y-component of the force which contributes to the tension.

### Other comments
- I wish I knew these in high school. So much better sana.

---
## Section 2: Code

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import math

### Problem 1: (Circuit Analysis)


Consider a long chain of resistors wired up like this:

<img src="reschain.png" alt="Alt text" width=95%>

All the resistors have the same resistance~$R$.  The power rail at the top is at voltage~$V_+=5$V.  The problem is to find the voltages $V_1...V_N$ at the internal points in the circuit.

1. **Using Ohm's law and the Kirchhoff current law**, which says that the total net current flow out of (or into) any junction in a circuit must be zero, show that the voltages $V_1\ldots V_N$ satisfy the equations

$$3V_1 - V_2 - V_3 = V_+ $$
$$-V_1 + 4V_2 - V_3 - V_4 = V_+ $$
      
$$-V_{i-2} - V_{i-1} + 4V_i - V_{i+1} - V_{i+2} = 0 $$

$$-V_{N-3} - V_{N-2} + 4V_{N-1} - V_N = V_- = 0 $$
$$-V_{N-2} - V_{N-1} + 3V_N = V_- = 0 $$

Note: The assignment of $V_-$ is to make the equations symmetric

2. Express these equations in vector form~$A\vec{v} = \vec{w}$ and find the values of the matrix~$A$ and the vector~$\vec{w}$.

3. Write a program to solve for the values of the~$V_i$ when there are $N=6$ internal junctions with unknown voltages.  (Hint: All the values of $V_i$ should lie between zero and $5$V.  If they don't, something is wrong.)

4. Now repeat your calculation for the case where there are $N=10\,000$ internal junctions.  This part is not possible using standard tools like the *solve* function.  You need to make use of the fact that the matrix~$\vec{A}$ is banded.  

In [2]:
# This is representing the matirx:
# This portion solves by brute force
# For n = 6
A = [[3,-1,-1,0,0,0], [-1,4,-1,-1,0,0],[-1,-1,4,-1,-1,0],[0,-1,-1,4,-1,-1],[0,0,-1,-1,4,-1],[0,0,0,-1,-1,3]]
w = [5,5,0,0,0,0]

In [3]:
# Performing the np.linalg.solve
x = np.linalg.solve(A,w)
print(x)

[3.7254902  3.43137255 2.74509804 2.25490196 1.56862745 1.2745098 ]


Then, 
+ V1 = 3.7254902
+ V2 = 3.43137255
+ V3 = 2.74509804
+ V4 = 2.25490196
+ V5 = 1.56862745
+ V6 = 1.2745098

In [4]:
# We now do this for N = 10000
# The challenge is to represent the N x N array
# Yet, I am informed that the more efficient way to do this would be
# Using the scipy.sparse.linalg.spsolve (Dr. Lim mentioned sparse in lec)
# Hence, we use it
from scipy import sparse
from scipy.sparse.linalg import spsolve

In [5]:
# We now set up the A and W matrices
def generateArrayA(N):
    mainDiag = np.full(N, 4)
    mainDiag[0] = 3
    mainDiag[-1] = 3
    
    diag1 = np.full(N-1, -1)
    diag2 = np.full(N-2, -1)
    
    # Initiating the sparse matrix
    diagonals = [diag2, diag1, mainDiag, diag1, diag2]
    offsets = [-2,-1,0,1,2]
    A = sparse.diags(diagonals, offsets, shape=(N,N), format="csc")
    
    return A.toarray()

def generateArrayB(N):
    x = np.zeros(N, dtype=float)
    x[0] = 5
    x[1] = 5
    return x

In [6]:
# to verify that the above works:
A_10000 = generateArrayA(10000)
w_10000 = generateArrayB(10000)

In [19]:
# We then solve the system
y = spsolve(A_10000,w_10000)
print(y)

# If it warns, just run the cell again, the error will not show na po

[4.99888228e+00 4.99861842e+00 4.99802841e+00 ... 1.97158611e-03
 1.38158071e-03 1.11772227e-03]


#### Problem 1 code summary

This program translates the resistor network into a matrix, which is easily solvable using np.linalg.solve. Manually, this problem would be solved by obtaining the solution to a system of linear equations. 

---

### Problem 2 (Statics)


Solve the following matrix problems. Set-up the linear equations that describe the problem then solve them 

Part 1: Suppose you have a hanging mass M supported by two ropes (angled by $\alpha$ and $\beta$ with respect to the ceiling). Make a function that calculates the tension on each of the ropes. Put your function in a python file and call it here in the notebook. Check that the answer by computer matches your own computations.

<del>
Part 2: 
Masses $m_1, m_2, m_3$, and $m_4$ lie on a $2.00$ [m] beam of negligible mass. They are located $0.20, 0.70, 1.10$, and $1.40$ [m] away from the left end of the beam. Determine the masses $m_1, m_2, m_3$, and $m_4$, given the following constraints: 

- The total mass is $8.00$ [kg]
- If a pivot is placed halfway, the beam will balance if a $1.05$ [kg] mass is placed on the right-end of the beam
- If a pivot is placed $1.20$ [m] away from the right end, then the beam would balance if a $550$ [g] mass is placed on the left end of the beam.
- If the system is split midway, the total mass on the left is $1.00$ [kg] heavier than the total mass on the right </del>

In [8]:
from angle import tension_ab

help(tension_ab)
tension_ab(np.pi/6,np.pi/3,10,9.8)

Help on function tension_ab in module angle:

tension_ab(a, b, m, g)
    The parameters are ([angle 1], [angle 2], [mass], [aceleration due to gravity])



(49.00000000000001, 84.870489570875)

#### Problem 2 code summary

This calculates the static equilibrium tensions in a two-rope suspension system by converting Newton's translational equilibrium equations ($\sum F_x = 0$ and $\sum F_y = 0$) into a $2 \times 2$ matrix system $A\vec{T} = \vec{b}$. I then used np.linalg.solve since it is only a 2 x 2 array multiplied to a 2 x 1 array.

---

## Section 3 (Notes)

### Some codes for solving matrix problems

* solving matrix equations ($\bf{A}\vec{x} = \vec{b}$) -> `np.linalg.solve`
* LU-decomposition ($\bf{A} = \bf{L}\bf{U}$) -> `scipy.linalg.lu`
* matrix inversion ($\bf{A}^{-1}$) -> `np.linalg.inv`
* QR-decomposition ($\bf{A} = \bf{Q}\bf{R}$) -> `scipy.linalg.qr`
* eigenvalue problem ($\bf{A}\vec{x} = \lambda \vec{x}$) -> `np.linalg.eig`

Solving matrix problems by themselves isn't difficult. There is an abundance of linear algebra libraries that are ready to use: from the battle-tested ones (the classic [BLAS](https://www.netlib.org/blas/) and [LAPACK](https://www.netlib.org/lapack/)), HPC tools for large problems ([ScaLAPACK](https://netlib.org/scalapack/)), to hardware-accelerated ones often with GPUs ([cuBLAS](https://docs.nvidia.com/cuda/cublas/index.html) and [cuSolver](https://docs.nvidia.com/cuda/cusolver/index.html) for NVIDIA). 

The hardest part is actually composing the matrix. How do you transform a physical problem into a linear algebra problem?

* Circuit: https://personal.math.vt.edu/embree/cmda3606/chapter2.pdf
* Embree [main notes](https://personal.math.vt.edu/embree/cmda3606notes.pdf) + [lab manual](https://personal.math.vt.edu/embree/labman.pdf)

#### Sample Usage (Testing different codes)

In [9]:
A = np.array([[1,0,2],[-2,-1,3],[0,3,5]])
B = np.array([[-2],[1],[-3]])

In [10]:
sigma_z = [[1,0],[0,-1]]
sigma_y = [[0,-1j],[1j,0]]
sigma_x = [[0,1],[1,0]]

In [11]:
ans = np.linalg.solve(sigma_x, sigma_y)
ans = np.linalg.inv(A) @ B


In [12]:
from scipy.linalg import *
p, l, u = lu(A)
p, l, u

(array([[0., 0., 1.],
        [1., 0., 0.],
        [0., 1., 0.]]),
 array([[ 1.        ,  0.        ,  0.        ],
        [-0.        ,  1.        ,  0.        ],
        [-0.5       , -0.16666667,  1.        ]]),
 array([[-2.        , -1.        ,  3.        ],
        [ 0.        ,  3.        ,  5.        ],
        [ 0.        ,  0.        ,  4.33333333]]))

### Importing functions from a python file

In [13]:
import samplePackage as sP

In [14]:
testMat = sP.createZerosArray(4,5)

In [15]:
testMat[1] = [1,2,3,4,5]

In [16]:
testMat

array([[0., 0., 0., 0., 0.],
       [1., 2., 3., 4., 5.],
       [0., 0., 0., 0., 0.],
       [0., 0., 0., 0., 0.]])

### np.roll

In [17]:
## Useful tips. Not required to solve the problems below!

# Suppose you have a Numpy array
x = np.zeros((4,4))
x[1,2] = 5
for row in x:
    print(row)

#rint(x[0,1:3])


# I can move all values to the left/right using roll
x_roll = np.roll(x, 1) # change 1 to -1, what happens? 
x_roll2 = np.roll(x, -1)
print(x_roll)

x_roll2 = np.roll(x, 2)
print(x_roll2)

[0. 0. 0. 0.]
[0. 0. 5. 0.]
[0. 0. 0. 0.]
[0. 0. 0. 0.]
[[0. 0. 0. 0.]
 [0. 0. 0. 5.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [5. 0. 0. 0.]
 [0. 0. 0. 0.]]
